# Notebook 03 — Case Retrieval (CBR Pidana Penadahan)

Notebook ini mengimplementasikan **Tahap Retrieval** dari pipeline Case-Based Reasoning (CBR) untuk putusan pidana penadahan.

Notebook ini **menggunakan langsung** output dari Notebook 02 (`data/processed/cases.csv`) dan **tidak** mengulang proses ekstraksi PDF, cleaning, metadata extraction, maupun labeling.

**Output notebook ini:**
- `data/eval/retrieval_metrics.csv`
- `data/eval/queries.json`
- `models/tfidf_vectorizer.pkl`
- `models/svm_model.pkl`
- Fungsi `retrieve(query, k=5)` yang siap dipakai pada Notebook 04 (Reuse)


## A. Import Library & Setup Folder

Menyiapkan seluruh library yang dibutuhkan dan memastikan folder output (`data/eval`, `models`) tersedia.

In [9]:
# ======================================
# CELL 1 - IMPORT & PROJECT PATH
# ======================================

import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# Root project
BASE_DIR = os.path.abspath(
    os.path.join(os.getcwd(), "..")
)

DATA_PROCESSED_PATH = os.path.join(
    BASE_DIR,
    "data",
    "processed",
    "cases.csv"
)

EVAL_DIR = os.path.join(
    BASE_DIR,
    "data",
    "eval"
)

MODELS_DIR = os.path.join(
    BASE_DIR,
    "models"
)

os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("=== PROJECT CONFIGURATION ===")
print("BASE_DIR            :", BASE_DIR)
print("DATA_PROCESSED_PATH :", DATA_PROCESSED_PATH)
print("EVAL_DIR            :", EVAL_DIR)
print("MODELS_DIR          :", MODELS_DIR)

assert os.path.exists(DATA_PROCESSED_PATH), (
    f"File tidak ditemukan:\n{DATA_PROCESSED_PATH}"
)

print("\n✅ Setup selesai.")

=== PROJECT CONFIGURATION ===
BASE_DIR            : /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN
DATA_PROCESSED_PATH : /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/data/processed/cases.csv
EVAL_DIR            : /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/data/eval
MODELS_DIR          : /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/models

✅ Setup selesai.


## A. Load Dataset

Membaca `data/processed/cases.csv` hasil Notebook 02, lalu melakukan validasi dasar:
- jumlah data
- jumlah label (`putusan`)
- missing values pada kolom penting


In [10]:
df = pd.read_csv(DATA_PROCESSED_PATH)

print("Jumlah baris data :", len(df))
print("Jumlah kolom       :", df.shape[1])
print()
print("Daftar kolom:")
print(list(df.columns))


Jumlah baris data : 50
Jumlah kolom       : 19

Daftar kolom:
['case_id', 'file_name', 'nomor_perkara', 'tanggal_putusan', 'jenis_perkara', 'pasal', 'terdakwa', 'ringkasan_fakta', 'amar_putusan', 'text_length', 'word_count', 'unique_words', 'putusan', 'text_full', 'txt_file', 'status', 'alasan_gagal', 'len_raw', 'len_clean']


In [11]:
# Validasi jumlah label
print("Distribusi label kolom 'putusan':")
print(df["putusan"].value_counts())
print()
print("Total label terisi:", df["putusan"].notna().sum(), "dari", len(df), "baris")


Distribusi label kolom 'putusan':
putusan
Sedang    27
Ringan    16
Berat      7
Name: count, dtype: int64

Total label terisi: 50 dari 50 baris


In [12]:
# Validasi missing values pada kolom-kolom kunci
kolom_kunci = ["case_id", "nomor_perkara", "putusan", "text_full"]
missing_report = df[kolom_kunci].isna().sum()

print("Missing values pada kolom kunci:")
print(missing_report)

assert df["case_id"].isna().sum() == 0, "Ada case_id yang kosong, periksa Notebook 02."
assert df["text_full"].isna().sum() == 0, "Ada text_full yang kosong, periksa Notebook 02."
assert df["putusan"].isna().sum() == 0, "Ada label putusan yang kosong, periksa Notebook 02."

print()
print("Validasi data: AMAN. Tidak ada missing value pada kolom kunci.")


Missing values pada kolom kunci:
case_id          0
nomor_perkara    0
putusan          0
text_full        0
dtype: int64

Validasi data: AMAN. Tidak ada missing value pada kolom kunci.


In [13]:
# Pastikan text_full bertipe string (menghindari error TF-IDF akibat NaN/float)
df["text_full"] = df["text_full"].astype(str)
df = df.reset_index(drop=True)

df.head(3)


,case_id,file_name,nomor_perkara,tanggal_putusan,jenis_perkara,pasal,terdakwa,ringkasan_fakta,amar_putusan,text_length,word_count,unique_words,putusan,text_full,txt_file,status,alasan_gagal,len_raw,len_clean
0,case_001,case_001.pdf,250/pid.b/,12 mei 2013,Pidana,480 ke -1 kuhpidana .mahkamah agung tersebut ;...,sugito,"menimbang, bahwa putusan pengadilan tinggi ban...",perkara direktori putusan mahkamah agung repu...,20146,2732,619,Sedang,direktori putusan mahkamah agung republik indo...,case_001.txt,SUKSES,NaN,20410,20146
1,case_002,case_002.pdf,103/pid.b/2014/pn,20 oktober 2014,Pidana,480,srihardono panggilan sri,"menimbang, bahwa putusan pengadilan negeri mua...",", menolak permohonan kasasi dari pemohon kasas...",34221,4678,848,Berat,direktori putusan mahkamah agung republik indo...,case_002.txt,SUKSES,NaN,34412,34221
2,case_003,case_003.pdf,15/pid.b/2023/pn,3 desember 2022,Pidana,480 ayat (1),maimunah,"menimbang, bahwa terdakwa diajukan ke persidan...",NaN,82285,10733,1387,Ringan,direktori putusan mahkamah agung republik indo...,case_003.txt,SUKSES,NaN,82982,82285


## B. Train Test Split

Membagi dataset menjadi data train dan test dengan `stratify` pada kolom `putusan` agar proporsi label (Ringan/Sedang/Berat) tetap terjaga di kedua subset.


In [14]:
X = df["text_full"]
y = df["putusan"]

X_train_text, X_test_text, y_train, y_test, idx_train, idx_test = train_test_split(
    X,
    y,
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Jumlah data train:", len(X_train_text))
print("Jumlah data test :", len(X_test_text))


Jumlah data train: 40
Jumlah data test : 10


In [15]:
print("Distribusi label TRAIN:")
print(y_train.value_counts())
print()
print("Distribusi label TEST:")
print(y_test.value_counts())


Distribusi label TRAIN:
putusan
Sedang    22
Ringan    13
Berat      5
Name: count, dtype: int64

Distribusi label TEST:
putusan
Sedang    5
Ringan    3
Berat     2
Name: count, dtype: int64


## C. TF-IDF Vectorization

TF-IDF dibentuk dari kolom `text_full`. Vectorizer di-*fit* hanya pada data **train** untuk menghindari data leakage, kemudian digunakan untuk men-transform data test.

Vectorizer ini juga yang akan dipakai oleh fungsi `retrieve()` pada bagian F dan disimpan untuk Notebook 04.


In [16]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words=None,
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

print("Shape TF-IDF train:", X_train_tfidf.shape)
print("Shape TF-IDF test :", X_test_tfidf.shape)


Shape TF-IDF train: (40, 5000)
Shape TF-IDF test : (10, 5000)


## D. SVM Retrieval Model

Melatih `LinearSVC` menggunakan representasi TF-IDF dari data train. Model ini berfungsi sebagai komponen klasifikasi pendukung retrieval (memprediksi label `putusan` dari sebuah kasus/query baru).


In [17]:
svm_model = LinearSVC(random_state=42)
svm_model.fit(X_train_tfidf, y_train)

print("Training SVM selesai.")
print("Jumlah fitur TF-IDF yang digunakan:", X_train_tfidf.shape[1])
print("Kelas yang dipelajari model:", list(svm_model.classes_))


Training SVM selesai.
Jumlah fitur TF-IDF yang digunakan: 5000
Kelas yang dipelajari model: ['Berat', 'Ringan', 'Sedang']


## E. Evaluasi Klasifikasi

Mengukur performa model SVM pada data test menggunakan Accuracy, Precision, Recall, dan F1-score (rata-rata `weighted` karena label tidak seimbang), lalu menyimpan hasilnya ke `data/eval/retrieval_metrics.csv`.


In [27]:
y_pred = svm_model.predict(X_test_tfidf)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print("Accuracy :", round(acc, 4))
print("Precision:", round(prec, 4))
print("Recall   :", round(rec, 4))
print("F1-score :", round(f1, 4))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))




Accuracy : 0.5
Precision: 0.25
Recall   : 0.5
F1-score : 0.3333

Classification Report:
              precision    recall  f1-score   support

       Berat       0.00      0.00      0.00         2
      Ringan       0.00      0.00      0.00         3
      Sedang       0.50      1.00      0.67         5

    accuracy                           0.50        10
   macro avg       0.17      0.33      0.22        10
weighted avg       0.25      0.50      0.33        10



In [19]:
print("Confusion Matrix:")
labels_order = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels_order)
cm_df = pd.DataFrame(cm, index=labels_order, columns=labels_order)
print(cm_df)


Confusion Matrix:
        Berat  Ringan  Sedang
Berat       0       0       2
Ringan      0       0       3
Sedang      0       0       5


In [20]:
metrics_df = pd.DataFrame([{
    "model": "LinearSVC",
    "kernel": "linear",
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1_score": f1,
    "n_train": len(X_train_text),
    "n_test": len(X_test_text),
}])

metrics_path = os.path.join(EVAL_DIR, "retrieval_metrics.csv")
metrics_df.to_csv(metrics_path, index=False)

print("Metrics disimpan ke:", metrics_path)
metrics_df


Metrics disimpan ke: /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/data/eval/retrieval_metrics.csv


,model,kernel,accuracy,precision,recall,f1_score,n_train,n_test
0,LinearSVC,linear,0.5,0.25,0.5,0.333333,40,10


## F. TF-IDF Retrieval Function

Fungsi `retrieve(query, k=5)` melakukan:
1. Preprocessing sederhana pada query (lowercase + strip)
2. Transform query menggunakan `tfidf_vectorizer` yang sudah di-*fit*
3. Menghitung cosine similarity terhadap seluruh dokumen pada dataset (`df`)
4. Mengambil top-k kasus dengan similarity tertinggi

Fungsi ini menggunakan TF-IDF dari **seluruh dataset** (`X` lengkap, bukan hanya train) agar retrieval pada Notebook 04 dapat mengambil kasus dari semua data yang tersedia.


In [21]:
# TF-IDF matrix untuk SELURUH dataset (basis pencarian retrieval)
X_full_tfidf = tfidf_vectorizer.transform(df["text_full"])

def preprocess_query(query: str) -> str:
    """Preprocessing sederhana untuk query teks."""
    return str(query).strip().lower()


def retrieve(query: str, k: int = 5) -> pd.DataFrame:
    """
    Mengambil k kasus paling mirip dengan query menggunakan TF-IDF + cosine similarity.

    Parameters
    ----------
    query : str
        Teks query (misalnya ringkasan fakta kasus baru).
    k : int
        Jumlah kasus teratas yang diambil.

    Returns
    -------
    pd.DataFrame dengan kolom: case_id, nomor_perkara, putusan, similarity
    diurutkan dari similarity tertinggi ke terendah.
    """
    clean_query = preprocess_query(query)
    query_vec = tfidf_vectorizer.transform([clean_query])

    sims = cosine_similarity(query_vec, X_full_tfidf).flatten()

    top_idx = np.argsort(sims)[::-1][:k]

    result = df.loc[top_idx, ["case_id", "nomor_perkara", "putusan"]].copy()
    result["similarity"] = sims[top_idx]
    result = result.reset_index(drop=True)

    return result


print("Fungsi retrieve() siap digunakan.")


Fungsi retrieve() siap digunakan.


## G. Uji Retrieval Manual

Menguji fungsi retrieve() dengan beberapa query contoh terkait kasus penadahan


In [32]:
# ======================================
# CELL - TEST RETRIEVAL
# ======================================

# Mengambil 5 kasus sebagai query uji
sample_queries = (
    df.sample(n=5, random_state=42)
      .reset_index(drop=True)
)

for i, row in sample_queries.iterrows():

    query = row["ringkasan_fakta"]

    print("=" * 80)
    print(f"Query {i+1}")
    print(f"True Label : {row['putusan']}")
    print(f"Query      : {query[:150]}...")

    hasil = retrieve(query, k=5)

    print("\nTop-5 Similar Cases")
    print(hasil)

    print()

Query 1
True Label : Sedang
Query      : menimbang, bahwa terdakwa diajukan kepersidangan dengan dakwaansebagai berikut :bahwa ia terdakwa agus nadi als dok bin alwi, pada hari jum attanggal ...

Top-5 Similar Cases
    case_id              nomor_perkara putusan  similarity
0  case_014  101/pid.b/2023/pn.plgdemi  Sedang    0.453662
1  case_005             32/pid/2023/pt  Sedang    0.343974
2  case_021          163/pid.b/2022/pn  Sedang    0.261546
3  case_026          224/pid.b/2022/pn  Ringan    0.238630
4  case_024          191/pid.b/2021/pn  Ringan    0.237288

Query 2
True Label : Sedang
Query      : menimbang, bahwa terdakwa diajukan ke persidangan oleh penuntut umum didakwa berdasarkan surat dakwaan sebagai berikut : bahwa terdakwa eko suryono al...

Top-5 Similar Cases
    case_id               nomor_perkara putusan  similarity
0  case_040           883/pid.b/2023/pn  Sedang    0.274389
1  case_044  1020/pid.b/2015/pn.bdgdemi  Ringan    0.211687
2  case_001                  250/

## H. Build queries.json

Membuat dataset query uji (`queries.json`) yang akan digunakan pada tahap **Case Solution Reuse** dan **Model Evaluation**.

Query diambil langsung dari beberapa kasus pada **case base**, yaitu menggunakan kolom **ringkasan_fakta** sehingga setiap query memiliki **ground truth** berupa label **putusan**.

File ini akan disimpan pada:

`data/eval/queries.json`

dan digunakan kembali pada Notebook 04 dan Notebook 05 agar seluruh proses evaluasi menggunakan query yang sama secara konsisten.

In [33]:
# ======================================
# CELL H - BUILD QUERIES.JSON
# ======================================

# Mengambil 5 kasus sebagai query evaluasi
sample_queries = (
    df.sample(n=5, random_state=42)
      .reset_index(drop=True)
)

queries_data = []

for i, row in sample_queries.iterrows():

    queries_data.append({

        "query_id": i + 1,

        "query_text": row["ringkasan_fakta"],

        "ground_truth": row["putusan"],

        "case_id": row["case_id"],

        "nomor_perkara": row["nomor_perkara"]

    })

queries_path = os.path.join(EVAL_DIR, "queries.json")

with open(queries_path, "w", encoding="utf-8") as f:
    json.dump(
        queries_data,
        f,
        indent=4,
        ensure_ascii=False
    )

print("=" * 60)
print("queries.json berhasil disimpan")
print(queries_path)
print("=" * 60)

# Preview isi file
with open(queries_path, "r", encoding="utf-8") as f:
    print(f.read())

queries.json berhasil disimpan
/Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/data/eval/queries.json
[
    {
        "query_id": 1,
        "query_text": "menimbang, bahwa terdakwa diajukan kepersidangan dengan dakwaansebagai berikut :bahwa ia terdakwa agus nadi als dok bin alwi, pada hari jum attanggal 21 oktober 2022 sekira pukul 18.00 wib atau setidak-tidaknya dalamtahun 2022, bertempat di jalan balap sepeda lr.muhajirin 4 rt.08 kelurahanlorok pakjo kecamatan ilir barat i kota palembang atau setidak-tidaknya padasuatu tempat yang masih termasuk dalam daerah hukum pengadilan negeripalembang yang berwenang memeriksa dan mengadili perkara ini, membeli,menyewa, menukar, menerima gadai, menerima hadiah, atau untuk menarikkeuntungan, menjual, menyewakan, menukarkan, menggadaikan, mengangkut,menyimpan, atau menyembunyikan sesuatu benda, yang diketahuinya atau putusan nomor 101/pid.b/2023/pn plg\fdirektori putusan mahkamah agung republik indonesia",
        "ground_tr

## I. Simpan Model

Menyimpan `tfidf_vectorizer` dan `svm_model` ke folder `models/` menggunakan `joblib.dump()`, agar dapat dimuat ulang pada Notebook 04 tanpa perlu melatih ulang.


In [24]:
tfidf_path = os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl")
svm_path = os.path.join(MODELS_DIR, "svm_model.pkl")

joblib.dump(tfidf_vectorizer, tfidf_path)
joblib.dump(svm_model, svm_path)

print("Model tersimpan:")
print("-", tfidf_path)
print("-", svm_path)


Model tersimpan:
- /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/models/tfidf_vectorizer.pkl
- /Users/asfaahmad/Codingan/Semester 6/Penalaran Komputer/CBR_PENADAHAN/models/svm_model.pkl


In [30]:
print("Accuracy :", round(acc, 4))
print()

print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

Accuracy : 0.5

              precision    recall  f1-score   support

       Berat       0.00      0.00      0.00         2
      Ringan       0.00      0.00      0.00         3
      Sedang       0.50      1.00      0.67         5

    accuracy                           0.50        10
   macro avg       0.17      0.33      0.22        10
weighted avg       0.25      0.50      0.33        10



In [31]:
import os

print("retrieval_metrics.csv :",
      os.path.exists(os.path.join(EVAL_DIR,
      "retrieval_metrics.csv")))

print("queries.json :",
      os.path.exists(os.path.join(EVAL_DIR,
      "queries.json")))

print("tfidf_vectorizer.pkl :",
      os.path.exists(os.path.join(MODELS_DIR,
      "tfidf_vectorizer.pkl")))

print("svm_model.pkl :",
      os.path.exists(os.path.join(MODELS_DIR,
      "svm_model.pkl")))

retrieval_metrics.csv : True
queries.json : True
tfidf_vectorizer.pkl : True
svm_model.pkl : True


## Ringkasan Notebook 03

- Dataset `cases.csv` dari Notebook 02 berhasil dimuat dan divalidasi.
- Data dibagi train/test (80/20) dengan stratifikasi label `putusan`.
- TF-IDF (`max_features=5000`, `ngram_range=(1,2)`) berhasil dibangun.
- Model `LinearSVC` dilatih dan dievaluasi (Accuracy, Precision, Recall, F1-score) → disimpan ke `data/eval/retrieval_metrics.csv`.
- Fungsi `retrieve(query, k=5)` berhasil diimplementasikan dan diuji dengan 5 query manual.
- `data/eval/queries.json` berisi 7 query uji dengan `ground_truth`.
- Model dan vectorizer disimpan ke `models/tfidf_vectorizer.pkl` dan `models/svm_model.pkl`.

Notebook ini **siap digunakan** sebagai basis Tahap 4 (Reuse) pada Notebook 04, dengan memuat ulang:
```python
tfidf_vectorizer = joblib.load("models/tfidf_vectorizer.pkl")
svm_model = joblib.load("models/svm_model.pkl")
```
dan menggunakan kembali fungsi `retrieve()` di atas.
